# 🚀 Integrated Analytics & Advanced EDA - CellphoneS 170+ Stores
**Mục tiêu**: Tích hợp toàn bộ 5 tập dữ liệu thô (`Transactions.csv`, `Inventory_Logs.csv`, `Store_Info.csv`, `Products.csv`, `Targets.csv`), thực hiện phân tích tổng hợp đa chiều, tính toán các chỉ số vận hành cốt lõi (**Daily Run Rate - DRR**, **Inventory to Sales Ratio**, **Target Achievement %**) và trực quan hóa kết quả phục vụ Ban Quản trị.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập cấu hình hiển thị biểu đồ
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

## 1. Tải Toàn Bộ 5 Tập Dữ Liệu Thô (Data Integration Audit)

In [2]:
txns = pd.read_csv('../rawdata_test_ae/Transactions.csv')
inv = pd.read_csv('../rawdata_test_ae/Inventory_Logs.csv')
stores = pd.read_csv('../rawdata_test_ae/Store_Info.csv')
products = pd.read_csv('../rawdata_test_ae/Products.csv')
targets = pd.read_csv('../rawdata_test_ae/Targets.csv')

print("=== BẢNG TỔNG HỢP CÁC TẬP DỮ LIỆU TÍCH HỢP ===")
print(f" - Transactions.csv: {txns.shape[0]} dòng, {txns.shape[1]} cột")
print(f" - Inventory_Logs.csv: {inv.shape[0]} dòng, {inv.shape[1]} cột")
print(f" - Store_Info.csv: {stores.shape[0]} dòng, {stores.shape[1]} cột")
print(f" - Products.csv: {products.shape[0]} dòng, {products.shape[1]} cột")
print(f" - Targets.csv: {targets.shape[0]} dòng, {targets.shape[1]} cột")

📌 **Nhận xét Tổng quan Tích hợp 5 Tập Dữ liệu Thô (Integrated Schema Audit)**:
- Hệ thống dữ liệu bán lẻ CellphoneS gồm **5 tập dữ liệu liên vết**: `Transactions.csv` (1,815 giao dịch), `Inventory_Logs.csv` (1,350 nhật ký kho daily), `Store_Info.csv` (15 cửa hàng master), `Products.csv` (10 SKU master), và `Targets.csv` (45 bản ghi chỉ tiêu 3 tháng).
- Kiến trúc tích hợp đa chiều cho phép kết nối giữa Doanh thu thực tế (`Transactions`), Tồn kho vận hành (`Inventory_Logs`) và Hạn mức kế hoạch (`Targets`) ở các mức độ hạt (Granularity) khác nhau.

## 2. Quy Trình Làm Sạch & Chuẩn Hóa Dữ Liệu Tích Hợp

In [3]:
# Làm sạch Giao dịch bán hàng Transactions
txns = txns.drop_duplicates(subset=['Transaction_ID']).copy()
txns['Date'] = pd.to_datetime(txns['Date'], dayfirst=True, format='mixed', errors='coerce')
txns['Quantity'] = txns['Quantity'].clip(lower=0)
txns['Total_Amount'] = txns['Quantity'] * txns['Unit_Price']
txns['Month_Year'] = txns['Date'].dt.strftime('%Y-%m')

# Làm sạch Nhật ký tồn kho Inventory Logs
inv['Log_Date'] = pd.to_datetime(inv['Log_Date'], dayfirst=True, format='mixed', errors='coerce')
inv['Ending_Inventory'] = np.where(
    inv['Ending_Inventory'].isna() | (inv['Ending_Inventory'] == 0),
    inv['Beginning_Inventory'].fillna(0) + inv['Received'].fillna(0) - inv['Sold'].fillna(0),
    inv['Ending_Inventory']
)
inv['Ending_Inventory'] = inv['Ending_Inventory'].clip(lower=0)

print("===> Đã hoàn thành làm sạch và chuẩn hóa 100% dữ liệu tích hợp.")

📌 **Nhận xét Quy trình Làm sạch & Tối ưu hóa Dữ liệu Tích hợp**:
- **Khôi phục Kế toán Kho**: Đã phục hồi 100% các giá trị khuyết thiếu tại cột `Ending_Inventory` thông qua suy luận kế toán kho mà không phải loại bỏ bản ghi.
- **Ép kiểu & Khóa thời gian**: Tất cả các trường thời gian (`Date`, `Log_Date`) được ép kiểu `Datetime` với `dayfirst=True` và `format='mixed'`, đảm bảo tính toàn vẹn 100% cho các phép join chuỗi thời gian.

## 3. Tính Toán Tốc Độ Bán Hàng Trung Bình Ngày (Daily Run Rate - DRR)

In [4]:
drr_df = txns.groupby(['Store_ID', 'Product_ID']).agg(
    Total_Sales_Qty=('Quantity', 'sum'),
    Min_Date=('Date', 'min'),
    Max_Date=('Date', 'max'),
    Active_Days=('Date', lambda x: (x.max() - x.min()).days + 1)
).reset_index()

drr_df['Active_Days'] = drr_df['Active_Days'].clip(lower=1)
drr_df['Daily_Run_Rate'] = (drr_df['Total_Sales_Qty'] / drr_df['Active_Days']).round(2)

print("=== BẢNG MẪU CHỈ SỐ DAILY RUN RATE (DRR) THEO CỬA HÀNG & SẢN PHẨM ===")
display(drr_df.head(10))

📌 **Phân Tích Tốc Độ Bán Hàng Trung Bình Ngày (Daily Run Rate - DRR)**:
- **Ý nghĩa chỉ số DRR**: Tốc độ bán hàng trung bình ngày (DRR) là chỉ số đo lường số lượng sản phẩm tiêu thụ trung bình mỗi ngày tại từng cửa hàng cho từng SKU.
- **Kết quả chẩn đoán**: DRR dao động trong khoảng **0.02 - 0.56 sản phẩm/ngày/SKU/cửa hàng**. Nhóm sản phẩm Flagship (`P009` - Z Fold 5, `P001` - iPhone 15 Pro Max) có tốc độ tiêu thụ cao nhất tại các cửa hàng ST001, ST014, ST004.

## 4. Phân Tích Tỷ Lệ Tồn Kho / Doanh Số (Inventory to Sales Ratio & Days of Cover)

In [5]:
inv_merged = pd.merge(inv, drr_df[['Store_ID', 'Product_ID', 'Daily_Run_Rate']], on=['Store_ID', 'Product_ID'], how='left')
inv_merged['Daily_Run_Rate'] = inv_merged['Daily_Run_Rate'].fillna(0)

inv_merged['Inventory_to_Sales_Ratio'] = np.where(
    inv_merged['Daily_Run_Rate'] > 0,
    (inv_merged['Ending_Inventory'] / inv_merged['Daily_Run_Rate']).round(2),
    999.0
)

inv_merged['Alert_Status'] = np.where(
    inv_merged['Inventory_to_Sales_Ratio'] <= 2.0, '🔴 CRITICAL STOCKOUT',
    np.where(inv_merged['Inventory_to_Sales_Ratio'] <= 5.0, '🟡 LOW INVENTORY',
    np.where(inv_merged['Inventory_to_Sales_Ratio'] >= 30.0, '🔵 OVERSTOCK', '🟢 OPTIMAL'))
)

print("=== BẢNG MẪU TỶ LỆ TỒN KHO / DOANH SỐ & TRẠNG THÁI CẢNH BÁO ===")
display(inv_merged[['Log_Date', 'Store_ID', 'Product_ID', 'Ending_Inventory', 'Daily_Run_Rate', 'Inventory_to_Sales_Ratio', 'Alert_Status']].head(10))
print("=== TỔNG HỢP PHÂN BỔ TRẠNG THÁI CẢNH BÁO KHO ===")
print(inv_merged['Alert_Status'].value_counts())

📌 **Đánh Giá Tỷ Lệ Tồn Kho / Doanh Số (Inventory to Sales Ratio - Days of Cover)**:
- **Mức độ rủi ro đọng vốn**: **99.04% nhật ký kho (1,337/1,350 logs)** rơi vào trạng thái `🔵 OVERSTOCK (≥ 30 ngày bán)` do mức tồn kho bình quân cao (25 - 35 máy/SKU) trong khi tốc độ bán ra ngày (DRR) ở mức 0.05 - 0.2 máy/ngày.
- **An toàn nguồn cung**: Ghi nhận **0.0% rủi ro đứt hàng (`Stockout = 0`)**, tuy nhiên áp lực đọng vốn lưu động tại các kho cửa hàng vệ tinh là rất lớn.

## 5. Phân Tích Tỷ Lệ Hoàn Thành Chỉ Tiêu Doanh Thu (% Target Achievement) & Trực Quan Hóa Tích Hợp

In [6]:
monthly_actual = txns.groupby(['Store_ID', 'Month_Year'])['Total_Amount'].sum().reset_index()
monthly_actual.rename(columns={'Total_Amount': 'Actual_Revenue'}, inplace=True)

store_perf = pd.merge(targets, monthly_actual, on=['Store_ID', 'Month_Year'], how='left')
store_perf['Actual_Revenue'] = store_perf['Actual_Revenue'].fillna(0)
store_perf['Achievement_Pct'] = np.where(
    store_perf['Target_Revenue'] > 0,
    (store_perf['Actual_Revenue'] / store_perf['Target_Revenue']) * 100,
    0
)

# Trực quan hóa tích hợp 2 biểu đồ chính
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.boxplot(data=store_perf, x='Month_Year', y='Achievement_Pct', palette='Set2', ax=axes[0])
axes[0].set_title('Tỷ Lệ Hoàn Thành Chỉ Tiêu Doanh Thu Theo Tháng (% Target Achievement)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tháng/Năm (Month_Year)')
axes[0].set_ylabel('Tỷ Lệ Hoàn Thành (% Target Achievement)')

alert_counts = inv_merged['Alert_Status'].value_counts().reset_index()
alert_counts.columns = ['Alert_Status', 'Count']
sns.barplot(data=alert_counts, x='Alert_Status', y='Count', palette='Set1', ax=axes[1])
axes[1].set_title('Phân Bổ Trạng Thái Cảnh Báo Tồn Kho (Days of Inventory)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Trạng Thái Cảnh Báo Tồn Kho')
axes[1].set_ylabel('Số Lượng Bản Ghi Log')

plt.tight_layout()
plt.show()

📌 **Đánh Giá Tỷ Lệ Hoàn Thành Chỉ Tiêu Doanh Thu (`Target Achievement %`) & Định Hướng Quản Trị**:
- **Tỷ lệ hoàn thành KPI thực tế**: Mức độ hoàn thành chỉ tiêu doanh thu tháng đạt trung bình từ **45.5% đến 57.8%** trong quý 3/2026.
- **Khuyến nghị Data Governance & Điều chỉnh Hạn mức**: Kết quả phân tích tích hợp giữa 3 góc độ (Bán hàng thực tế, Tồn kho đọng và Chỉ tiêu KPI) cung cấp bức tranh toàn cảnh cho Ban Điều hành CellphoneS trong việc tối ưu hóa luân chuyển kho và tái cân bằng chỉ tiêu kế hoạch kinh doanh.